# Assignment: Regression

In this assignment you will apply the regression model from class to a vehicle of your choosing and use it to find listings that are priced unusually well — or unusually poorly — relative to comparable vehicles.

Please read these instructions carefully before you begin:

- **AI use**: You may use AI to *ask questions or do code completion*. Do not use AI to write whole blocks of code and don't copy and paste from a web UI. The purpose of these restrictions are to ensure that you are in the driver's seat on the conceptual pieces of the assignment. 
- **Questions**: Look for cells marked with `Q:`. These require written answers in Markdown. Do not leave them blank — your interpretation is the most important part of this assignment.
- **Output**: Ensure your notebook runs top-to-bottom without errors. Remove stray debug code before submitting.

When you've finished your work on this notebook, restart the kernel, clear the outputs, and run all cells to ensure everything works as expected. Then commit and push your changes to GitHub. As always, let me know if you have any questions. 

### Getting Your Data

This week you'll apply regression to **real-world car listings data** pulled through our self-serve system.

Here is a link to [the self-serve form](https://docs.google.com/forms/d/e/1FAIpQLSdQQMShcp2OaFkdOclqkJN5hmWjWsowEte62WrglmW1S61Vkw/viewform?usp=header).

After submitting the form you'll receive a tab-delimited text file in a zip via email. Save it to this folder and update `file_path` in the next cell before running the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
%matplotlib inline

---

## Part 1: Data Preparation

### Loading and Cleaning

Run the same cleaning pipeline from previous assignments. It is provided for you here.

In [ ]:
file_path = 'results.zip'   # replace with your actual filename--you can read zips directly
listings = pd.read_csv(file_path) # Feel free to use a better, more specific name! 

listings['time_posted']          = pd.to_datetime(listings['time_posted'], errors='coerce')
listings['year_from_time_posted'] = listings['time_posted'].dt.year
listings['year']       = pd.to_numeric(listings['year'],       errors='coerce').astype('Int64')
listings['odometer']   = pd.to_numeric(listings['odometer'],   errors='coerce').astype('Int64')
listings['num_images'] = pd.to_numeric(listings['num_images'], errors='coerce').astype('Int64')
listings['price']      = pd.to_numeric(listings['price'],      errors='coerce')
listings['latitude']   = pd.to_numeric(listings['latitude'],   errors='coerce')
listings['longitude']  = pd.to_numeric(listings['longitude'],  errors='coerce')

listings = listings.drop_duplicates(subset='url')
for col in ['make', 'model', 'location', 'title', 'fuel', 'drive',
            'transmission', 'paint', 'type', 'condition']:
    listings[col] = listings[col].str.lower()

print(listings.shape)
listings.head()

### Missing Data

In this next cell, address any missing data or clearly incorrect data in the following columns: `price`, `odometer`, `location`, `time_posted`, `year`, `make`, and `model`. Missing or incorrect data can either be dropped or imputed.  


In [ ]:
# Your code here

Q: Did you find missing data in your data set? How did you handle them? 

A: <!-- Your answer here -->

Q: Did you find any extreme data points in the data? How did you handle them?

A: <!-- Your answer here -->

### Filtering and Feature Engineering

Filter to clean- or lien-title listings with valid prices, odometer readings, and ages. Then create three model features:
- `age`: The age of the car at the time it was listed
- `log_odometer`: natural log of the odometer reading
- `in_focal_location`: 1 if the listing is from your chosen city, 0 otherwise. If you've chosen data from just one location, you can skip this step.

Set the three variables below before running the rest of the notebook.

In [ ]:
# Your code here

Q: How many listings do you have after filtering? How many are in your focal location?

A: <!-- Your answer here -->

Q: Which make/model and focal location did you choose? Why did you pick this one?

A: <!-- Your answer here -->

---

## Part 2: Exploring Your Data

Before fitting a model, look at the variables individually and together.

### Price Distribution

Create a histogram of `price` for your vehicle, and compute the mean, median, standard deviation, and IQR.

In [ ]:
# Your code here: histogram of 'price'


In [ ]:
# Your code here: compute mean, median, std, and IQR of price
# Hint: pd.Series.describe() gives you most of these; IQR = Q3 - Q1


Q: Is the price distribution skewed, or roughly symmetric? Is the mean higher or lower than the median, and what does that tell you?

A: <!-- Your answer here -->

### Price vs. Odometer

Create two scatterplots of `price` vs. odometer: one using the raw odometer reading, and one using `log_odometer`. In each, put odometer on the x-axis and price on the y-axis.

In [ ]:
# Your code here: price vs. raw odometer


In [ ]:
# Your code here: price vs. log_odometer


Q: How did the log transformation change what you can see in the scatterplot? Which version shows a clearer linear relationship?

A: <!-- Your answer here -->

### Price vs. Car Age

Create a scatterplot of `price` vs. `car_age`.

In [ ]:
# Your code here: price vs. car_age


Q: Does price decline steadily with age, or is the pattern more complicated? What do you notice?

A: <!-- Your answer here -->

---

## Part 3: Fitting the Model

Fit the same model structure from the in-class exercise:

$$\text{price} = \beta_0 + \beta_1 \cdot \text{car\_age} + \beta_2 \cdot \text{log\_odometer} + \beta_3 \cdot \text{in\_focal\_location} + \varepsilon$$

Use `smf.ols()` with the formula string `'price ~ car_age + log_odometer + in_focal_location'` and `data=vehicle`.

In [ ]:
# Your code here: fit the model and save as results
results = ___

In [ ]:
results.summary()

Q: Using the ideas from class, interpret the `car_age` coefficient in plain English. Include the sign and magnitude. Confused by the output above? This is an excellent time to ask AI for clarification.

A: <!-- Your answer here -->

Q: What does the `in_focal_location` coefficient tell you? Does your chosen city tend to have higher or lower prices than other cities in the dataset, after controlling for age and mileage?

A: <!-- Your answer here -->

Q: What is R-squared for your model? What does it tell you about how well age and mileage explain price variation for your vehicle? Does it surprise you?

A: <!-- Your answer here -->

--- 

In the code below, recreate the code from the exercise that creates new data for you to call predict on. I've recreated the code here for ease of reference: 
```
miles = [50_000, 100_000, 150_000, 200_000, 250_000]
car_age = 2026 - 2010

scenarios = pd.DataFrame({
    'car_age': [car_age] * len(miles),
    'log_odometer': np.log(miles),
    'in_minneapolis': [1] * len(miles),
    'odometer_miles': miles
})

scenarios['estimated_price'] = results.predict(scenarios)
scenarios['diff_from_prev'] = scenarios['estimated_price'].diff()
```

You'll need to update the code for your particular data and the scenarios you find interesting. See the exercise for how I displayed the results. 

In [ ]:
# Your code here

Q: Interpret the relationship between `odometer` (or `log_odometer`) and `price`. Build your interpretation based on quantities like miles and dollars. It's not enough to say "price goes down \$XXX with every one unit increase in `log_odometer`. Write for other humans! Since there's a log relationship you can just interpret a few results from the scenarios explored above. 

A: <!-- Your answer here -->

---

## Part 4: Finding Good and Bad Deals

A listing is a **good deal** if it's priced well *below* what the model would predict for a vehicle with those same characteristics. A listing is potentially **overpriced** if it's well *above* the model prediction.

We flag deals using **residuals**: the difference between a listing's actual price and the model's predicted price.

$$\text{residual} = \text{actual price} - \text{predicted price}$$

Listings with residuals below −2 standard deviations are flagged as good deals. Listings above +2 standard deviations are potentially overpriced.

In [ ]:
# Compute predicted prices and residuals for every listing. Here is some pseudo code to help
# vehicle['predicted_price'] = results.predict(vehicle)
# vehicle['residual']        = vehicle['price'] - vehicle['predicted_price']
#
# resid_std = vehicle['residual'].std()
# print(f'Residual std dev: ${resid_std:,.0f}')

In [ ]:
# Plot the distribution of residuals


Q: Describe the distribution of the residuals. 

A: <!-- Your answer here -->

### Good Deals

Filter to listings where the residual is below −2 standard deviations, then show the results sorted by residual (most underpriced first).

In [ ]:
# Your code here: filter to good deals (residual < -2 * resid_std) and sort by residual


Q: Look at your top good deal — the listing with the lowest residual. What year is it? How many miles does it have? Does the price seem legitimately low, or could there be a data quality issue (e.g., a price entered in thousands, or a placeholder value)?

A: <!-- Your answer here -->

### Potentially Overpriced Listings

Filter to listings where the residual is above +2 standard deviations.

In [ ]:
# Your code here: filter to overpriced listings (residual > 2 * resid_std) and sort


Q: Look at one of the overpriced listings. Can you think of a plausible reason it might be listed above the model's prediction? (Think about what the model *doesn't* know about a listing.)

A: <!-- Your answer here -->

---

## Part 5: Reflection

- What was your familiarity with regression before this assignment? 
- Undoubtedly you have questions about regression given our quick treatment. What question stands out for you?  
- What is one important factor that affects used car prices that this model completely ignores?
- If you were actually shopping for your chosen vehicle, would you trust this model's good-deal flags? Why or why not?
- How long did this assignment take you?

<!-- Your answers here -->